# 00. Colab setup

Mounts Google Drive (data, cache and checkpoints survive session restarts), clones the repo, installs the package with `uv` and downloads Yambda-50M.

The preinstalled Colab torch/numpy are kept: dependency lower bounds in `pyproject.toml` are loose on purpose.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
os.environ["RECSYS_DATA_ROOT"] = "/content/drive/MyDrive/tlab-recsys"
os.makedirs(os.environ["RECSYS_DATA_ROOT"], exist_ok=True)

In [ ]:
# Private repo: needs a GitHub token stored in Colab Secrets as GH_TOKEN (Contents: read-only).
import os
import sys

from google.colab import userdata

token = (userdata.get("GH_TOKEN") or "").strip()
if not token:
    raise RuntimeError("Colab secret GH_TOKEN is empty, see notebooks/README.md")
REPO_URL = f"https://{token}@github.com/miont/tlab-2026-recsys-task2.git"

!git clone -q {REPO_URL} /content/repo || git -C /content/repo pull -q
if not os.path.exists("/content/repo/pyproject.toml"):
    raise RuntimeError("git clone failed: check that GH_TOKEN can read the repository")
%cd /content/repo
!pip install -q uv
!uv pip install --system -q -e ".[tracking]"

# The editable install's .pth file is only read at interpreter startup,
# so make the package importable in the already running kernel.
sys.path.insert(0, "/content/repo/src")

In [ ]:
!yambda-download --out $RECSYS_DATA_ROOT/raw

In [ ]:
import torch

from recsys_rl.config import load_config
from recsys_rl.utils import get_device

cfg = load_config("configs/base.yaml")
gpu = torch.cuda.get_device_name() if torch.cuda.is_available() else ""
print(torch.__version__, get_device(), gpu)
print(cfg.raw_dir, cfg.processed_dir, cfg.run_dir)

## Optional: audio embeddings for the gold RM

`embeddings.parquet` has all 7.7M Yambda tracks (~13.8 GB), more than a free Drive holds. The RM needs only the vocabulary items (~50 MB), so the file is downloaded to the Colab VM disk, the vocabulary cache `processed/50m/audio_embeddings.npy` is built from it and copied to Drive. Run once, after the processed data exists (`01_eda` or `02_baseline`); the VM copy disappears with the runtime.

In [ ]:
import dataclasses
import shutil

from huggingface_hub import hf_hub_download

from recsys_rl.data import prepare
from recsys_rl.reward_model import load_audio_embeddings

local = dataclasses.replace(cfg, data_root="/content/yambda_local")  # VM disk, not Drive
hf_hub_download("yandex/yambda", "embeddings.parquet", repo_type="dataset", local_dir=local.raw_dir)
emb = load_audio_embeddings(local, prepare(cfg).items)
cfg.processed_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(
    local.processed_dir / "audio_embeddings.npy", cfg.processed_dir / "audio_embeddings.npy"
)
print(emb.shape, "->", cfg.processed_dir / "audio_embeddings.npy")